# Task 1: Data Cleaning & Preprocessing
**Dataset:** Netflix Movies and TV Shows
**Objective:** Prepare the raw Netflix dataset for analysis by finding and fixing missing values, duplicates, inconsistent formats, and transforming columns so that later tasks (EDA, modeling) can use a clean file.

**Workflow:** Import data → Inspect → Handle missing/duplicate records → Fix formats → Transform columns → Save clean dataset

# Step 1: Import libraries and load the dataset

Import the required libraries and load the provided `Dataset.csv` file with Pandas. I keep an untouched copy called `df_raw` so that the original data can be compared with the cleaned working dataframe.

In [65]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv("../data/Dataset.csv")

# Remove any accidental index columns created during CSV export
df = df_raw.loc[:, ~df_raw.columns.str.startswith("Unnamed")].copy()

print("Raw shape:", df_raw.shape)
print("Working shape:", df.shape)

display(df.head())

Raw shape: (8790, 10)
Working shape: (8790, 10)


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


## Step 2: Inspect the dataset structure

I inspect the data types, number of non-null values, and basic statistics before making any changes. This gives me an initial view of the dataset and shows which columns may require cleaning or type conversion.


In [66]:
df.info()

print("\nSummary statistics:")
display(df.describe(include="all").T)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8790 non-null   object
 1   type          8790 non-null   object
 2   title         8790 non-null   object
 3   director      8790 non-null   object
 4   country       8790 non-null   object
 5   date_added    8790 non-null   object
 6   release_year  8790 non-null   int64 
 7   rating        8790 non-null   object
 8   duration      8790 non-null   object
 9   listed_in     8790 non-null   object
dtypes: int64(1), object(9)
memory usage: 686.8+ KB

Summary statistics:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
show_id,8790,8790,s8786,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type,8790,2,Movie,6126,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,8790,8787,9-Feb,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
director,8790,4528,Not Given,2588,NaN,NaN,NaN,NaN,NaN,NaN,NaN
country,8790,86,United States,3240,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date_added,8790,1713,1/1/2020,110,NaN,NaN,NaN,NaN,NaN,NaN,NaN
release_year,8790.0,NaN,NaN,NaN,2014.183163,8.825466,1925.0,2013.0,2017.0,2019.0,2021.0
rating,8790,14,TV-MA,3205,NaN,NaN,NaN,NaN,NaN,NaN,NaN
duration,8790,220,1 Season,1791,NaN,NaN,NaN,NaN,NaN,NaN,NaN
listed_in,8790,513,"Dramas, International Movies",362,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Step 3: Check for empty columns

Before handling missing values inside the dataset, I check whether any columns are completely empty. Completely empty columns do not provide useful information and should not be carried into the cleaned dataset.


In [67]:
empty_columns = df.columns[df.isna().all()].tolist()

print("Completely empty columns:", empty_columns)

if empty_columns:
    df = df.drop(columns=empty_columns)

print("Shape after removing empty columns:", df.shape)

Completely empty columns: []
Shape after removing empty columns: (8790, 10)


## Step 4: Standardize text and handle missing-value placeholders

The source uses values such as `"Not Given"` to represent missing information. I first remove unnecessary whitespace from text columns.

I then replace known placeholders only in columns where they represent missing data. I avoid applying the replacement globally because values such as `"Unknown"` or `"NA"` could potentially be legitimate data in a title or another field.


In [68]:
text_cols = df.select_dtypes(include="object").columns

# Remove leading and trailing whitespace from text fields
df[text_cols] = df[text_cols].apply(lambda c: c.str.strip())

placeholders = ["Not Given", "N/A", "NA", "", "Unknown"]

missing_value_cols = [
    "director",
    "country",
    "rating",
    "date_added",
    "duration"
]

for col in missing_value_cols:
    df[col] = df[col].replace(placeholders, np.nan)

## Step 5: Verify that valid titles were not converted to missing values

The raw file contains no missing titles. I therefore check the original `title` column for values that look like missing-value placeholders.

This check is important because applying a global placeholder replacement could incorrectly turn a valid title into `NaN`.


In [69]:
title_placeholders = ["Unknown", "NA", "N/A", "Not Given"]

title_check = df_raw[
    df_raw["title"].isin(title_placeholders)
]

print("Placeholder-like titles found in raw data:")
display(title_check)

print("Missing titles in working dataframe:", df["title"].isna().sum())

Placeholder-like titles found in raw data:


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
1336,s1468,Movie,Unknown,Jaume Collet-Serra,United Kingdom,1/1/2021,2011,PG-13,113 min,"Action & Adventure, Thrillers"


Missing titles in working dataframe: 0


## Step 6: Profile missing values

I calculate the number and percentage of missing values in each column. Looking at both measures makes it easier to identify which missing-value problems are substantial and which are negligible.


In [70]:
missing = df.isna().sum()

missing_summary = pd.DataFrame({
    "missing": missing,
    "percent": (missing / len(df) * 100).round(2)
}).sort_values("missing", ascending=False)

display(missing_summary)

,missing,percent
director,2588,29.44
country,287,3.27
type,0,0.00
show_id,0,0.00
title,0,0.00
date_added,0,0.00
release_year,0,0.00
rating,0,0.00
duration,0,0.00
listed_in,0,0.00


## Step 7: Check for misplaced duration values

Some versions of this dataset have durations like `"74 min"` entered in the `rating` column. I therefore check the `rating` column for values that look like movie durations before standardizing the ratings.

If any are found, they should be moved to the `duration` column rather than discarded.

In [71]:
misplaced_duration_mask = (
    df["rating"]
    .astype("string")
    .str.contains(r"\bmin\b", case=False, na=False)
)

print(
    "Durations found in rating column:",
    misplaced_duration_mask.sum()
)

if misplaced_duration_mask.any():
    df.loc[misplaced_duration_mask, "duration"] = (
        df.loc[misplaced_duration_mask, "rating"]
    )

    df.loc[misplaced_duration_mask, "rating"] = np.nan

print(
    "Remaining misplaced durations:",
    df["rating"]
    .astype("string")
    .str.contains(r"\bmin\b", case=False, na=False)
    .sum()
)

Durations found in rating column: 0
Remaining misplaced durations: 0


## Step 7 Decision: No misplaced durations were found

The check found 0 duration-like values in the `rating` column, so no records needed to be moved or corrected in this step.

## Step 8: Inspect duplicate content records

Because `show_id` is unique for each Netflix record, checking duplicates across every column would not identify records that represent the same title with different IDs.

I therefore check for duplicates using the content columns while excluding `show_id`. I also inspect repeated combinations of title, type, and release year before deciding whether the records should be removed.

In [72]:
content_cols = df.columns.drop("show_id")

print(
    "Duplicates ignoring show_id:",
    df.duplicated(subset=content_cols).sum()
)

dups = df[
    df.duplicated(
        subset=["title", "type", "release_year"],
        keep=False
    )
].copy()

dups = dups.sort_values(
    ["title", "type", "release_year"]
)

display(dups)

Duplicates ignoring show_id: 4


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
2925,s3963,Movie,15-Aug,Swapnaneel Jayakar,India,3/29/2019,2019,TV-14,124 min,"Comedies, Dramas, Independent Movies"
4261,s5967,Movie,15-Aug,Swapnaneel Jayakar,India,3/29/2019,2019,TV-14,124 min,"Comedies, Dramas, Independent Movies"
3285,s4523,Movie,22-Jul,Paul Greengrass,Norway,10/10/2018,2018,R,144 min,"Dramas, Thrillers"
4260,s5966,Movie,22-Jul,Paul Greengrass,Norway,10/10/2018,2018,R,144 min,"Dramas, Thrillers"
393,s3997,TV Show,9-Feb,NaN,Pakistan,3/20/2019,2018,TV-14,1 Season,"International TV Shows, TV Dramas"
537,s5965,TV Show,9-Feb,NaN,Pakistan,3/20/2019,2018,TV-14,1 Season,"International TV Shows, TV Dramas"
2590,s3372,Movie,Consequences,Ozan Açıktan,Turkey,10/25/2019,2014,TV-MA,106 min,"Dramas, International Movies, Thrillers"
4718,s6530,Movie,Consequences,Ozan Açıktan,Turkey,10/25/2019,2014,TV-MA,106 min,"Dramas, International Movies, Thrillers"


## Step 9: Remove confirmed duplicate content records

The duplicate inspection identified four pairs of records that contain the same content information and differ only by `show_id`.

The affected titles are `15-Aug`, `22-Jul`, `9-Feb`, and `Consequences`. The date-like titles are retained exactly as supplied by the source because they appear to be legitimate titles that may have been converted to date-like values by spreadsheet software. I do not guess or replace their original titles.

Since the duplicate records contain identical content fields, I keep the first `show_id` and remove the repeated content records.

In [73]:
df = df.drop_duplicates(
    subset=content_cols,
    keep="first"
).copy()

print(
    "Shape after removing duplicate content records:",
    df.shape
)

print(
    "Duplicate show_id:",
    df["show_id"].duplicated().sum()
)

print(
    "Duplicate content records:",
    df.duplicated(
        subset=content_cols
    ).sum()
)

Shape after removing duplicate content records: (8786, 10)
Duplicate show_id: 0
Duplicate content records: 0


### Note on date-like titles

A small number of titles, including `15-Aug`, `22-Jul`, and `9-Feb`, appear as date-like values in the provided dataset. These values were retained exactly as supplied because they may be legitimate titles that were converted to date formats by spreadsheet software. No titles were manually reconstructed or guessed.

## Step 10: Standardize rating values

The ratings contain both `NR` and `UR`, which represent titles without a standard rating. I merge both values into a single `Not Rated` category.

I also fill any remaining missing ratings with the same label so that the column has one consistent representation for unrated titles.


In [74]:
print("Ratings before standardization:")
print(df["rating"].value_counts(dropna=False))

df["rating"] = (
    df["rating"]
    .replace({
        "NR": "Not Rated",
        "UR": "Not Rated"
    })
    .fillna("Not Rated")
)

print("\nRatings after standardization:")
print(df["rating"].value_counts())

Ratings before standardization:
rating
TV-MA       3204
TV-14       2155
TV-PG        861
R            798
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64

Ratings after standardization:
rating
TV-MA        3204
TV-14        2155
TV-PG         861
R             798
PG-13         490
TV-Y7         333
TV-Y          306
PG            287
TV-G          220
Not Rated      82
G              41
TV-Y7-FV        6
NC-17           3
Name: count, dtype: int64


## Step 11: Handle remaining missing values

The missing director and country values represent information that was not provided in the source dataset. I keep these records rather than dropping them because removing thousands of rows would unnecessarily reduce the dataset.

I replace the missing values with explicit `Unknown` labels so that the records remain available for later analysis.


In [75]:
df["director"] = df["director"].fillna("Unknown")
df["country"] = df["country"].fillna("Unknown")

print("Missing values after treatment:")
print(df.isna().sum())

Missing values after treatment:
show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64


## Step 12: Convert `date_added` to datetime

The `date_added` column is stored as text in month/day/year format. I convert it to a proper datetime column so that it can be used for time-based analysis.

I also extract the year, month number, and month name because these features will be useful when analysing catalogue additions over time.


In [76]:
df["date_added"] = pd.to_datetime(
    df["date_added"],
    format="%m/%d/%Y",
    errors="coerce"
)

df["year_added"] = df["date_added"].dt.year
df["month_added"] = df["date_added"].dt.month
df["month_name"] = df["date_added"].dt.month_name()

assert df["date_added"].isna().sum() == 0, "Some dates failed to parse"

print("Date parsing check passed.")

display(
    df[
        [
            "date_added",
            "year_added",
            "month_added",
            "month_name"
        ]
    ].head()
)

Date parsing check passed.


,date_added,year_added,month_added,month_name
0,2021-09-25,2021,9,September
1,2021-09-24,2021,9,September
2,2021-09-24,2021,9,September
3,2021-09-22,2021,9,September
4,2021-09-24,2021,9,September


## Step 13: Convert month names to an ordered category

The month names are currently stored as text. If they are sorted alphabetically, months would appear in an order that does not follow the calendar.

I therefore convert `month_name` into an ordered categorical variable so that later analysis follows the correct January-to-December order.


In [77]:
months = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

df["month_name"] = pd.Categorical(
    df["month_name"],
    categories=months,
    ordered=True
)

print(df["month_name"].dtype)

category


## Step 14: Transform the duration column

The `duration` column contains two different units: minutes for movies and seasons for TV shows.

I extract the numerical value and unit, then create separate numeric columns for movie runtime and number of TV seasons. This makes the fields easier to use in statistical analysis and visualizations.


In [78]:
df["duration_value"] = (
    df["duration"]
    .str.extract(r"(\d+)", expand=False)
    .astype(float)
)

df["duration_unit"] = (
    df["duration"]
    .str.extract(r"([A-Za-z]+)", expand=False)
    .str.lower()
    .replace({"season": "seasons"})
)

df["movie_minutes"] = np.where(
    df["type"] == "Movie",
    df["duration_value"],
    np.nan
)

df["tv_seasons"] = np.where(
    df["type"] == "TV Show",
    df["duration_value"],
    np.nan
)

display(
    df[
        [
            "type",
            "duration",
            "duration_value",
            "duration_unit",
            "movie_minutes",
            "tv_seasons"
        ]
    ].head()
)

,type,duration,duration_value,duration_unit,movie_minutes,tv_seasons
0,Movie,90 min,90.0,min,90.0,NaN
1,TV Show,1 Season,1.0,seasons,NaN,1.0
2,TV Show,1 Season,1.0,seasons,NaN,1.0
3,Movie,91 min,91.0,min,91.0,NaN
4,Movie,125 min,125.0,min,125.0,NaN


## Step 15: Create clean country, genre, and director lists

The `country`, `listed_in`, and `director` columns can contain multiple comma-separated values. I split these fields into lists and remove extra spaces and empty entries.

For countries, `"Unknown"` represents missing information rather than a real country, so it is converted to an empty list. This ensures that unknown countries are not counted as actual countries in later analysis.

Directors are also split into individual names so that director-level analysis counts each credited person separately rather than treating joint credits as one name.

I also calculate the number of countries and genres associated with each title.

In [79]:
def split_clean(value):
    if pd.isna(value):
        return []
    return [item.strip() for item in value.split(",") if item.strip()]


df["country_list"] = df["country"].apply(
    lambda v: [] if v == "Unknown" else split_clean(v)
)

df["genre_list"] = df["listed_in"].apply(split_clean)

df["director_list"] = df["director"].apply(
    lambda v: [] if v == "Unknown" else split_clean(v)
)

df["n_countries"] = df["country_list"].str.len()
df["n_genres"] = df["genre_list"].str.len()
df["n_directors"] = df["director_list"].str.len()

display(
    df[
        [
            "country",
            "country_list",
            "n_countries",
            "listed_in",
            "genre_list",
            "n_genres",
            "director",
            "director_list",
            "n_directors"
        ]
    ].head()
)

,country,country_list,n_countries,listed_in,genre_list,n_genres,director,director_list,n_directors
0,United States,[United States],1,Documentaries,[Documentaries],1,Kirsten Johnson,[Kirsten Johnson],1
1,France,[France],1,"Crime TV Shows, International TV Shows, TV Act...","[Crime TV Shows, International TV Shows, TV Ac...",3,Julien Leclercq,[Julien Leclercq],1
2,United States,[United States],1,"TV Dramas, TV Horror, TV Mysteries","[TV Dramas, TV Horror, TV Mysteries]",3,Mike Flanagan,[Mike Flanagan],1
3,Brazil,[Brazil],1,"Children & Family Movies, Comedies","[Children & Family Movies, Comedies]",2,Bruno Garotti,[Bruno Garotti],1
4,United States,[United States],1,"Dramas, Independent Movies, International Movies","[Dramas, Independent Movies, International Mov...",3,Haile Gerima,[Haile Gerima],1


## Step 16: Check the ordering of genre lists

The `listed_in` values contain multiple genres separated by commas. Before assigning any genre as a "primary" genre, I check whether the values have an inherent order.

The check below compares each genre list with its alphabetically sorted version.

In [80]:
print(
    "Genre lists sorted A-Z:",
    (df["genre_list"].apply(lambda genres: genres == sorted(genres))).mean().round(3)
)

Genre lists sorted A-Z: 1.0


## Step 16 Decision: Do not create a primary genre

100% of genre lists are alphabetical, so the first genre carries no meaning. I did not create `primary_genre` and will use the exploded `genre_list` instead.

## Step 17: Check country and director fields before analysis

Before ranking countries and directors, I check how these fields are structured and whether there are any data-quality patterns that could affect the analysis.

**Countries:** `country_list` allows several countries per title, so I first check how many countries each title actually has. I then compare the exploded country counts with the counts from the original `country` field. If they are identical, the dataset contains one country per title and does not represent co-productions.

I also inspect a sample of titles labelled `"Pakistan"` because Pakistan ranks highly in the dataset and some sampled titles appear inconsistent with that country label.

**Directors:** Missing directors are stored as `"Unknown"`, which becomes an empty list in `director_list`. Exploding the list therefore excludes unknown directors automatically. Joint director credits are split into individual names so each credited director is counted separately.

In [81]:
# --- Country structure ---

print("Number of countries per title:")
display(
    df["n_countries"]
    .value_counts()
    .sort_index()
)

country_counts = (
    df["country_list"]
    .explode()
    .dropna()
    .value_counts()
)

print("\nTop 10 countries:")
display(country_counts.head(10))

# Compare exploded country counts with counts from the original country field
plain_counts = (
    df.loc[df["country"] != "Unknown", "country"]
    .value_counts()
)

print(
    "Exploded counts identical to plain counts:",
    country_counts.to_dict() == plain_counts.to_dict()
)


# --- Sanity check on Pakistan ---

print("\nRandom sample of titles labelled Pakistan:")

display(
    df.loc[
        df["country"] == "Pakistan",
        ["title", "type", "director", "listed_in", "release_year"]
    ].sample(15, random_state=1)
)


# --- Directors ---

director_counts = (
    df["director_list"]
    .explode()
    .dropna()
    .value_counts()
)

print("\nTop 10 directors (Unknown excluded):")
display(director_counts.head(10))

Number of countries per title:


n_countries
0     287
1    8499
Name: count, dtype: int64


Top 10 countries:


country_list
United States     3240
India             1056
United Kingdom     638
Pakistan           420
Canada             271
Japan              259
South Korea        214
France             213
Spain              182
Mexico             138
Name: count, dtype: int64

Exploded counts identical to plain counts: True

Random sample of titles labelled Pakistan:


,title,type,director,listed_in,release_year
7579,Criminal: France,TV Show,Unknown,"Crime TV Shows, International TV Shows, TV Dramas",2019
357,El desconocido,TV Show,Unknown,"Crime TV Shows, International TV Shows, Spanis...",2019
47,Bread Barbershop,TV Show,Unknown,"Kids' TV, TV Comedies",2020
387,Super Monsters Furever Friends,Movie,Unknown,Children & Family Movies,2019
34,Chicago Party Aunt,TV Show,Unknown,TV Comedies,2021
517,Pac's Scary Halloween,Movie,Unknown,Movies,2016
442,Salt Fat Acid Heat,TV Show,Unknown,Docuseries,2018
6780,Trash Truck,TV Show,Unknown,Kids' TV,2021
4688,Chupan Chupai,Movie,Mohsin Ali,"Comedies, International Movies",2017
35,Squid Game,TV Show,Unknown,"International TV Shows, TV Dramas, TV Thrillers",2021



Top 10 directors (Unknown excluded):


director_list
Rajiv Chilaka          23
Jan Suter              21
Alastair Fothergill    19
Raúl Campos            19
Suhas Kadav            16
Marcus Raboy           16
Jay Karas              15
Cathy Garcia-Molina    13
Jay Chapman            12
Youssef Chahine        12
Name: count, dtype: int64

### Investigating the Pakistan country label

The sample contains several titles whose country association appears inconsistent with the `"Pakistan"` label. However, the high rate of missing directors cannot be used on its own as proof that the country label is incorrect because director information is missing for most TV Shows in the dataset.

To control for this effect, I compare the share of titles with an Unknown director within each content type.

In [82]:
# Compare missing-director rates across countries and content types

known = df[df["country"] != "Unknown"].copy()

known["unknown_director"] = known["director"].eq("Unknown")

by_type = (
    known
    .groupby(
        ["type", "country"],
        observed=True
    )["unknown_director"]
    .agg(
        share="mean",
        n="size"
    )
    .reset_index()
    .query("n >= 50")
    .round(3)
)

for t in ["TV Show", "Movie"]:
    print(f"\nUnknown-director share among {t}s (n >= 50):")

    display(
        by_type[
            by_type["type"] == t
        ]
        .sort_values("share", ascending=False)
        .drop(columns="type")
        .head(10)
    )

print("\nOverall share by type:")

display(
    df.groupby(
        "type",
        observed=True
    )["director"]
    .apply(lambda s: s.eq("Unknown").mean())
    .round(3)
)


Unknown-director share among TV Shows (n >= 50):


,country,share,n
115,Pakistan,0.997,349
79,Australia,0.981,53
103,Japan,0.942,172
84,Canada,0.940,84
124,South Korea,0.933,165
134,United States,0.923,845
94,France,0.877,65
128,Taiwan,0.873,71
98,India,0.864,81
133,United Kingdom,0.853,251



Unknown-director share among Movies (n >= 50):


,country,share,n
47,Pakistan,0.746,71
5,Brazil,0.050,60
71,United Kingdom,0.047,387
20,Germany,0.025,79
72,United States,0.025,2395
69,Turkey,0.025,81
16,Egypt,0.018,109
11,China,0.017,59
9,Canada,0.016,187
63,Spain,0.016,129



Overall share by type:


type
Movie      0.028
TV Show    0.906
Name: director, dtype: float64

### Interpreting the type-controlled results

The results show that missing director information is strongly associated with content type.

About 90.6% of TV Shows have an Unknown director, compared with only 2.8% of Movies. Therefore, the very high missing-director rate among Pakistan-labelled TV Shows is not sufficient evidence that the country label is incorrect.

However, Pakistan-labelled Movies show a much stronger anomaly: 74.6% have an Unknown director compared with 2.8% of Movies overall.

Combined with the sampled titles that appear inconsistent with Pakistan, this provides evidence that some country labels may be inaccurate. However, the dataset itself does not provide enough information to determine the correct country for those records.

## Step 17 Decision: Country and director handling

- **One country per title:** every title has either 0 or 1 country, so the dataset contains no multi-country or co-production records. Exploding `country_list` gives the same counts as the original `country` field.

- **Pakistan label is questionable:** Pakistan ranks 4th with 420 titles. A sample of 15 Pakistan-labelled titles contains several titles whose country association appears inconsistent with the label, including *Squid Game*, *Criminal: France*, and *Chicago Party Aunt*. This suggests that some country values may be inaccurate.

- **Director completeness provides supporting evidence, but not proof:** 95.5% of Pakistan-labelled titles have an Unknown director, compared with 29.4% overall. However, Pakistan is heavily weighted toward TV Shows, where 90.6% of titles overall have an Unknown director. When controlled by content type, 99.7% of Pakistan-labelled TV Shows and 74.6% of Pakistan-labelled Movies have Unknown directors. The movie figure is particularly unusual because only 2.8% of Movies overall have an Unknown director.

- **Decision:** I cannot verify the correct country values from this dataset alone, so I keep the country values as provided rather than manually relabelling them. Later country analysis will therefore be described as **"in this dataset"**, and Pakistan will be treated as a sensitivity check rather than an unquestioned country-level finding.

- **Unknown countries:** 287 titles have no country and are excluded from country rankings.

- **Directors:** Unknown directors are excluded from director rankings. Joint director credits are split into individual names so each credited director is counted separately. Because director information is missing for most TV Shows, director rankings should be interpreted mainly as movie-oriented.

## Step 18: Create an audience group

The original rating categories are useful but can be grouped into broader audience segments for analysis.

I classify titles into `Kids`, `Teens`, `Adults`, and `Not Rated`. The assignment of `TV-PG` and `PG` to the Kids group is a broad analytical choice rather than a claim that every title with these ratings is specifically children's content.

In [83]:
kids = [
    "TV-Y",
    "TV-Y7",
    "TV-Y7-FV",
    "TV-G",
    "G",
    "TV-PG",
    "PG"
]

teens = [
    "PG-13",
    "TV-14"
]

adults = [
    "R",
    "TV-MA",
    "NC-17"
]

df["audience"] = np.select(
    [
        df["rating"].isin(kids),
        df["rating"].isin(teens),
        df["rating"].isin(adults)
    ],
    [
        "Kids",
        "Teens",
        "Adults"
    ],
    default="Not Rated"
)

print(df["audience"].value_counts())

audience
Adults       4005
Teens        2645
Kids         2054
Not Rated      82
Name: count, dtype: int64


## Step 19: Convert categorical columns to `category` dtype

Columns with a small set of repeated values (`type`, `rating`, `audience`) are converted to the `category` type. This saves memory and signals that they are categorical variables.


In [84]:
for col in ["type", "rating", "audience"]:
    df[col] = df[col].astype("category")

df[["type", "rating", "audience"]].dtypes

type        category
rating      category
audience    category
dtype: object

## Step 20: Validate the relationship between content type and duration

Movies should have durations measured in minutes, while TV shows should have durations measured in seasons.

I use a cross-tabulation to check whether the transformed duration fields agree with the content type.


In [85]:
duration_check = pd.crosstab(
    df["type"],
    df["duration_unit"]
)

display(duration_check)

duration_unit,min,seasons
type,,
Movie,6123,0
TV Show,0,2663


## Step 21: Check year and month coverage

Check the years and months represented in the dataset to understand the time coverage before performing any time-based analysis.

The dataset ends in September 2021 rather than December 2021. Therefore, 2021 is an incomplete year. For forecasting in Task 4, complete years should be used for the main training period rather than treating 2021 as a full year.

In [86]:
print("Titles added by year:")
display(df["year_added"].value_counts().sort_index())

print("\nTitles added in 2021 by month:")
display(
    df.loc[df["year_added"] == 2021, "month_added"]
      .value_counts()
      .sort_index()
)

print("\nLatest date added:", df["date_added"].max())

Titles added by year:


year_added
2008       2
2009       2
2010       1
2011      13
2012       3
2013      11
2014      24
2015      82
2016     426
2017    1185
2018    1647
2019    2013
2020    1879
2021    1498
Name: count, dtype: int64


Titles added in 2021 by month:


month_added
1    132
2    109
3    112
4    188
5    132
6    207
7    257
8    178
9    183
Name: count, dtype: int64


Latest date added: 2021-09-25 00:00:00


## Step 21 Decision: Treat 2021 as partial and start the time series in 2016
The data ends on 25 September 2021, so both 2021 and September 2021 are incomplete.
Years 2008 to 2015 contain only 1 to 82 titles per year, and they only reflect titles
still on the catalogue in 2021, so they are not reliable for trend analysis.
For time-based tasks I will use a monthly series from January 2016 to August 2021,
the last complete month.

## Step 21 Validation: Confirm the latest available month

I verify the latest year and month represented in the dataset so that the forecasting boundary is based on the actual data rather than an assumption.

In [87]:
latest_date = df["date_added"].max()

print("Latest date:", latest_date)
print("Latest year:", latest_date.year)
print("Latest month:", latest_date.month)

Latest date: 2021-09-25 00:00:00
Latest year: 2021
Latest month: 9


## Step 22 Decision: Flag negative year gaps instead of removing them

There are 14 titles where the recorded `release_year` occurs after the year the title was added to Netflix.

Most of these cases are TV Shows, where `release_year` may reflect the latest season rather than the original release year. The two movie records (`Incoming` and `Hans Teeuwen: Real Rancour`) do not have the same explanation and may represent data-entry issues or pre-release listings.

Because the source data does not provide enough information to determine the exact cause, I do not delete these records. Instead, I flag them with `added_before_release` so they can be identified during later analysis.

In [88]:
df["years_to_add"] = df["year_added"] - df["release_year"]

early = df[
    df["years_to_add"] < 0
][
    ["title", "type", "release_year", "date_added", "years_to_add"]
]

print("Records where release year is after the date-added year:")
display(early.sort_values("years_to_add"))

print("\nType breakdown:")
display(early["type"].value_counts())

df["added_before_release"] = df["years_to_add"] < 0

print("\nNumber flagged:", df["added_before_release"].sum())

Records where release year is after the date-added year:


,title,type,release_year,date_added,years_to_add
5178,Jack Taylor,TV Show,2016,2013-03-31,-3
8223,Sense8,TV Show,2018,2016-12-23,-2
5138,Incoming,Movie,2019,2018-10-26,-1
3869,Hans Teeuwen: Real Rancour,Movie,2018,2017-07-01,-1
6992,Polly Pocket,TV Show,2021,2020-11-15,-1
7355,Love Is Blind,TV Show,2021,2020-02-13,-1
7436,Fuller House,TV Show,2020,2019-12-06,-1
6948,Hilda,TV Show,2021,2020-12-14,-1
7484,Maradona in Mexico,TV Show,2020,2019-11-13,-1
7517,BoJack Horseman,TV Show,2020,2019-10-25,-1



Type breakdown:


type
TV Show    12
Movie       2
Name: count, dtype: int64


Number flagged: 14


## Step 23: Inspect the day of month for titles added

Check whether titles are concentrated on particular days of the month.

A noticeable number of titles were added on the first day of the month, with 2,209 titles (25.14%) added on day 1. Day 15 is also unusually frequent, with 686 titles.

The concentration on specific days across many different months is consistent with scheduled licensing or catalogue-release cycles, where content may become available according to planned release schedules. However, this dataset alone cannot confirm the exact operational reason.

Because of these patterns, daily-level analysis should be treated cautiously. Year-level and month-level trends are more useful for understanding catalogue growth.

In [89]:
day_counts = df["date_added"].dt.day.value_counts().sort_index()

print("Titles added by day of month:")
display(day_counts)

day_1_count = (df["date_added"].dt.day == 1).sum()
day_15_count = (df["date_added"].dt.day == 15).sum()

print("\nTitles added on day 1:", day_1_count)
print(
    "Percentage added on day 1:",
    round(day_1_count / len(df) * 100, 2)
)

print("\nTitles added on day 15:", day_15_count)
print(
    "Percentage added on day 15:",
    round(day_15_count / len(df) * 100, 2)
)

Titles added by day of month:


date_added
1     2209
2      325
3      151
4      174
5      231
6      210
7      194
8      201
9      147
10     213
11     149
12     181
13     175
14     198
15     686
16     288
17     180
18     207
19     243
20     248
21     193
22     230
23     184
24     159
25     196
26     205
27     195
28     190
29     140
30     210
31     274
Name: count, dtype: int64


Titles added on day 1: 2209
Percentage added on day 1: 25.14

Titles added on day 15: 686
Percentage added on day 15: 7.81


## Step 24: Check unusually short movie runtimes

Movies with runtimes below 40 minutes are relatively short and may represent shorts, documentaries, specials, or other non-standard movie formats.

I do not remove these records because a short runtime is not evidence of an invalid record. Instead, I keep them and document the count for later analysis.

In [90]:
short_movies = df[
    (df["type"] == "Movie") &
    (df["movie_minutes"] < 40)
]

print(
    "Movies under 40 minutes:",
    len(short_movies)
)

display(
    short_movies[
        [
            "title",
            "release_year",
            "movie_minutes",
            "listed_in"
        ]
    ].head(20)
)

Movies under 40 minutes: 169


,title,release_year,movie_minutes,listed_in
13,Bridgerton - The Afterparty,2021,39.0,Movies
14,Bling Empire - The Afterparty,2021,36.0,Movies
15,Cobra Kai - The Afterparty,2021,34.0,Movies
16,The Circle - The Afterparty,2021,35.0,Comedies
80,My Heroes Were Cowboys,2021,23.0,Documentaries
154,To All the Boys: Always and Forever - The Afte...,2021,36.0,Movies
155,Trese After Dark,2021,36.0,"Anime Features, Documentaries"
183,Dad Stop Embarrassing Me - The Afterparty,2021,37.0,Comedies
209,Ginny & Georgia - The Afterparty,2021,35.0,"Comedies, Dramas"
235,Ya no estoy aquí: Una conversación entre Guill...,2020,15.0,Documentaries


## Step 25: Check for empty strings in the list columns

Splitting comma-separated values can sometimes create empty strings when the source contains stray commas.

I check both list columns after cleaning to make sure that no empty category values remain.


In [91]:
empty_country_items = (
    df["country_list"]
    .explode()
    .eq("")
    .sum()
)

empty_genre_items = (
    df["genre_list"]
    .explode()
    .eq("")
    .sum()
)

print("Empty strings in country_list:", empty_country_items)
print("Empty strings in genre_list:", empty_genre_items)

Empty strings in country_list: 0
Empty strings in genre_list: 0


## Step 26: Review remaining null values

At this stage, remaining null values should mainly come from fields that do not apply to a particular content type.

For example, a TV show does not have a movie runtime, while a movie does not have a number of TV seasons. These are structural nulls rather than missing data and should be left as `NaN`.


In [92]:
final_missing = df.isna().sum()

print("Columns with remaining null values:")
display(
    final_missing[
        final_missing > 0
    ]
)

Columns with remaining null values:


movie_minutes    2663
tv_seasons       6123
dtype: int64

## Step 27: Final validation

Before saving the cleaned dataset, I run a final set of checks covering row count, identifiers, duplicate content, missing titles, unknown values, and the cleaned list fields.

This provides a final quality check before the dataset is passed to the next tasks.


In [93]:
print("Final shape:", df.shape)

print("Duplicate show_id:", df["show_id"].duplicated().sum())
print("Duplicate content:", df.duplicated(subset=content_cols).sum())
print("Missing titles:", df["title"].isna().sum())

print("Unknown directors:", (df["director"] == "Unknown").sum())
print("Unknown countries:", (df["country"] == "Unknown").sum())
print("Not Rated:", (df["rating"] == "Not Rated").sum())

print(
    "Empty country list items:",
    df["country_list"].explode().eq("").sum()
)

print(
    "Empty genre list items:",
    df["genre_list"].explode().eq("").sum()
)

Final shape: (8786, 26)
Duplicate show_id: 0
Duplicate content: 0
Missing titles: 0
Unknown directors: 2587
Unknown countries: 287
Not Rated: 82
Empty country list items: 0
Empty genre list items: 0


## Step 28: Save the cleaned dataset

Save the cleaned dataframe in both CSV and Pickle formats.

The CSV file is the portable version that can be inspected or shared. The Pickle file preserves Python/Pandas data types, including categorical columns, but it should not be treated as a public data-sharing format because loading Pickle files can execute embedded Python objects.

In [94]:
df.to_pickle("../data/netflix_clean.pkl")
df.to_csv("../data/netflix_clean.csv", index=False)

print("Clean dataset saved successfully.")

Clean dataset saved successfully.


## Step 29: Reload and verify the saved datasets

The cleaned dataset was saved in both CSV and Pickle formats.

The CSV is useful as a portable tabular file, but CSV does not preserve Python-specific structures such as list columns or Pandas categorical dtypes. The Pickle file preserves these structures and will be used when continuing the analysis in Python.

I therefore verify the basic integrity of the CSV and the structure of the Pickle file separately.

In [95]:
# Verify the portable CSV output
df_csv = pd.read_csv("../data/netflix_clean.csv")

print("CSV shape:", df_csv.shape)
print("CSV columns:", len(df_csv.columns))
print(
    "CSV duplicate show_id:",
    df_csv["show_id"].duplicated().sum()
)
print(
    "CSV missing titles:",
    df_csv["title"].isna().sum()
)

# Verify the Python/Pandas Pickle output
df_pickle = pd.read_pickle("../data/netflix_clean.pkl")

print("\nPickle shape:", df_pickle.shape)
print("Pickle columns:", len(df_pickle.columns))

print("\nPickle list-column types:")
print("country_list:", type(df_pickle["country_list"].iloc[0]))
print("genre_list:", type(df_pickle["genre_list"].iloc[0]))
print("director_list:", type(df_pickle["director_list"].iloc[0]))

print("\nPickle categorical columns:")
print(
    df_pickle[["type", "rating", "audience"]].dtypes
)

print("\nPickle month_name dtype:")
print(df_pickle["month_name"].dtype)

CSV shape: (8786, 26)
CSV columns: 26
CSV duplicate show_id: 0
CSV missing titles: 0

Pickle shape: (8786, 26)
Pickle columns: 26

Pickle list-column types:
country_list: <class 'list'>
genre_list: <class 'list'>
director_list: <class 'list'>

Pickle categorical columns:
type        category
rating      category
audience    category
dtype: object

Pickle month_name dtype:
category


# Step 30: Final cleaning summary

The Netflix dataset was cleaned and prepared for analysis.

- Original dataset: 8,790 rows and 10 columns
- Final dataset: 8,786 rows and 26 columns
- 4 duplicate content records were removed while retaining unique `show_id` values
- Missing directors were represented as `Unknown`
- Missing countries were represented as `Unknown`
- `Unknown` was not treated as a real country when creating `country_list`
- `NR` and `UR` ratings were standardized to `Not Rated`
- `date_added` was converted to datetime and year/month fields were created
- 2021 was identified as an incomplete year ending in September
- Negative release-year gaps were retained and flagged rather than removed
- Duration was separated into movie minutes and TV seasons
- Country, genre, and director fields were converted into lists for later analysis
- Joint director credits were split so directors can be counted individually
- Genre order was verified as alphabetical, so no `primary_genre` was created
- Short movies were retained because short runtime does not by itself indicate invalid data
- No empty country or genre list items remained
- Audience groups were created as `Kids`, `Teens`, `Adults`, and `Not Rated`
- Categorical fields were converted to appropriate categorical data types

The cleaned dataset is saved as `netflix_clean.csv` for analysis and `netflix_clean.pkl` for preserving Pandas data types.

In [96]:
print("Final dataset shape:", df.shape)

print("\nFinal columns:")
print(len(df.columns))

print("\nUnknown directors:")
print((df["director"] == "Unknown").sum())

print("\nUnknown countries:")
print((df["country"] == "Unknown").sum())

print("\nNot Rated titles:")
print((df["rating"] == "Not Rated").sum())

print("\nAdded before release year:")
print(df["added_before_release"].sum())

print("\nMovies under 40 minutes:")
print(
    (
        (df["type"] == "Movie") &
        (df["movie_minutes"] < 40)
    ).sum()
)

print("\nStructural nulls:")
print("movie_minutes:", df["movie_minutes"].isna().sum())
print("tv_seasons:", df["tv_seasons"].isna().sum())

print("\nEmpty columns:")
print(
    df.columns[
        df.isna().all()
    ].tolist()
)

print("\nMisplaced durations in rating:")
print(
    df["rating"]
    .astype(str)
    .str.contains(r"\b\d+\s*min\b", regex=True)
    .sum()
)

Final dataset shape: (8786, 26)

Final columns:
26

Unknown directors:
2587

Unknown countries:
287

Not Rated titles:
82

Added before release year:
14

Movies under 40 minutes:
169

Structural nulls:
movie_minutes: 2663
tv_seasons: 6123

Empty columns:
[]

Misplaced durations in rating:
0
